# Comparação entre planilha Numbers e PDFs

Este notebook compara a planilha principal de turmas com os PDFs do DSC e dos cursos para localizar inconsistências de cadastro. A execução está organizada em células de preparação, células de regras individuais e uma célula final de geração do relatório.

**Arquivos lidos**
- `Dados/` como referência principal de entrada, mesmo quando `Dados` for um link simbólico ou um alias do macOS.
- Uma planilha `.numbers`, `.xlsx` ou `.csv` encontrada automaticamente a partir de `Dados/`.
- `Dados/DSC.pdf`
- `Dados/BCC_mat.pdf`
- `Dados/BCC_not.pdf`
- `Dados/BCD_not.pdf`
- `Dados/SIS_not.pdf`

**Estrutura das células de código**
- Célula 1: imports, constantes e configuração de exibição.
- Célula 2: funções auxiliares de leitura, normalização, extração dos PDFs e validações.
- Célula 3: localização da pasta `Dados/` e da planilha principal.
- Célula 4: leitura da planilha e extração das turmas dos PDFs.
- Célula 5: preparação das fontes combinadas usadas nas regras.
- Células seguintes: uma célula por regra do relatório, nesta ordem:
  - `A) __ Conflitos entre PDFs e Planilha (qtd.)`: `A1` turmas dos cursos PDFs ausentes, `A2` turmas do DSC ausentes, `A3` horário diferente, `A4` nome diferente, `A5` curso diferente.
  - `B) __ Coluna Caixa de Seleção (qtd.)`: `B1` concentrado não marcado, `B2` conflito de concentrado, `B3` EAD não marcado, `B4` conflito de `!DSC`, `B5` conflito de laboratório.
  - `C) __ Conflitos créditos x horas aula (qtd.)`: `C1` total de créditos ímpar, `C2` carga horária com zero, `C3` carga horária incompatível com créditos.
  - `D) __ Conflitos Professor (qtd.)`: `D1` turmas sem professor, `D2` professor diferente, `D3` conflitos de professor.
  - `E) __ Conflitos Espaço físico (qtd.)`: `E1` conflitos de espaço físico.
  - `F) __ Conflitos Horário no Semestre (qtd.)`: `F1` conflitos de semestre/fase/grupo no mesmo horário.
- Célula final: montagem do resumo agrupado, geração do Markdown e gravação de `Dados/comparacao.md`.

**Normalizações aplicadas**
- Na coluna `Código`, tanto na planilha Numbers quanto nos PDFs, códigos com ponto seguido de espaço são normalizados removendo somente o espaço após o ponto. Exemplo: `CMP. 0170. 01. 002-9` vira `CMP.0170.01.002-9`.
- Na comparação de horário da regra `A3`, o sufixo `C` depois do número do horário é desconsiderado. Exemplo: `Sex 1/2` e `Sex 1/2C` são tratados como iguais.

**Relatório gerado**
- `Dados/comparacao.md`, com resumo agrupado e tabelas Markdown por seção.


In [1]:
from __future__ import annotations

import re
import subprocess
import unicodedata
import warnings
from pathlib import Path

import pandas as pd
import pdfplumber
from IPython.display import Markdown, display
from numbers_parser import Document

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 200)

BASE_PATH = Path.cwd()
DADOS_REF = BASE_PATH / "Dados"
COURSE_PDFS = {
    "BCC_mat.pdf": "BCC-M",
    "BCC_not.pdf": "BCC-N",
    "BCD_not.pdf": "BCD-N",
    "SIS_not.pdf": "SIS-N",
}
COURSE_DAY_X = {"Seg": 389, "Ter": 421, "Qua": 452, "Qui": 484, "Sex": 515, "Sab": 547}
DSC_DAY_X = {"Seg": 523, "Ter": 554, "Qua": 582, "Qui": 616, "Sex": 647, "Sab": 677}
DAY_ORDER = {"Seg": 0, "Ter": 1, "Qua": 2, "Qui": 3, "Sex": 4, "Sab": 5}
LABORATORIOS_VALIDOS = {
    "CAMPUS2_G-004", "EFEX", "LAB_G-201", "LAB_G-206", "LAB_J-200", "LAB_N-109", "LAB_R-129",
    "LAB_S-212", "LAB_S-224", "LAB_S-301", "LAB_S-401", "LAB_S-403", "LAB_S-409", "LAB_S-410",
    "LAB_S-412", "LAB_S-413", "LAB_S-415", "LAB_S-427", "LAB_S-429", "LAB_S-430", "LAB_S-432", "LAB_T-208",
}

print(f"Notebook em: {BASE_PATH}")
print(f"Referência de dados: {DADOS_REF}")

Notebook em: /Users/daltonreis/GitHub/DSC/dsc/___Python
Referência de dados: /Users/daltonreis/GitHub/DSC/dsc/___Python/Dados


In [2]:
def strip_accents(value: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", value) if not unicodedata.combining(ch))


def normalize_space(value) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return re.sub(r"\s+", " ", str(value).replace("\xa0", " ")).strip()


def clean_placeholder(value) -> str:
    text = normalize_space(value)
    if strip_accents(text).lower() in {
        "_nao_", "_nao oferta_", "_psps", "_nao oferta", "_nao", "nao oferta", "nao",
    }:
        return ""
    return text


def normalize_code(value) -> str:
    text = normalize_space(value)
    if not text:
        return ""
    text = re.sub(r"\.\s+", ".", text)
    compact_text = text.replace(" ", "")
    match = re.search(r"([A-Z]{3}\.\d{4}\.\d{2}\.\d{3}-\d)", compact_text)
    return match.group(1) if match else text


def normalize_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    text = strip_accents(normalize_space(value)).lower()
    return text in {"true", "1", "sim", "yes", "x"}


def normalize_time_token(token: str) -> str:
    text = normalize_space(token).upper().replace("-", "/")
    text = text.replace(" / ", "/").replace("/ ", "/").replace(" /", "/")
    text = text.replace(" C", "C")
    return text


def sort_tokens(tokens):
    def key(item):
        day, token = item.split(":", 1)
        return (DAY_ORDER.get(day, 99), token)
    return tuple(sorted(set(tokens), key=key))


def compose_horario(tokens) -> str:
    ordered = sort_tokens(tokens)
    return "; ".join(f"{day} {token}" for day, token in (item.split(":", 1) for item in ordered))


def key_text(value) -> str:
    return strip_accents(normalize_space(value)).upper()


def assign_day(x0: float, day_map: dict[str, float]) -> str:
    return min(day_map, key=lambda day: abs(x0 - day_map[day]))


def cluster_page_words(page):
    words = sorted(page.extract_words(use_text_flow=False), key=lambda word: (word["top"], word["x0"]))
    clusters = []
    for word in words:
        if not clusters or abs(word["top"] - clusters[-1]["top"]) > 2.5:
            clusters.append({"top": word["top"], "words": [word]})
        else:
            clusters[-1]["words"].append(word)
    for cluster in clusters:
        cluster["words"] = sorted(cluster["words"], key=lambda word: word["x0"])
    return clusters


def looks_like_professor_parenthetical(value: str) -> bool:
    text = normalize_space(value).removesuffix(")").strip()
    if not text:
        return False
    if key_text(text) in {"EAD"}:
        return False
    return len(text.split()) >= 2


def extract_name_professor(parts):
    text = " ".join(parts).strip()
    if "(" in text:
        start = text.rfind("(")
        maybe_name = text[:start].strip()
        maybe_professor = text[start + 1:].removesuffix(")").strip()
        if maybe_name and looks_like_professor_parenthetical(maybe_professor):
            return maybe_name, maybe_professor
    return text, ""


def has_open_professor_parenthesis(text: str) -> bool:
    start = text.rfind("(")
    if start == -1:
        return False
    maybe_professor = text[start + 1:]
    return ")" not in maybe_professor and looks_like_professor_parenthetical(maybe_professor)


def extract_course_phase_code(words) -> tuple[str, str] | None:
    left_text = " ".join(word["text"] for word in words if word["x0"] < 120)
    left_text = re.sub(r"\.\s+", ".", left_text)
    compact_left = left_text.replace(" ", "")
    match = re.search(r"(\d+)([A-Z]{3}\.\d{4}\.\d{2}\.\d{3}-\d)", compact_left)
    if match:
        phase, code = match.groups()
        return phase, normalize_code(code)

    for word in words:
        compact_word = normalize_code(word["text"]).replace(" ", "")
        match = re.match(r"^(\d+)([A-Z]{3}\.\d{4}\.\d{2}\.\d{3}-\d)$", compact_word)
        if match:
            phase, code = match.groups()
            return phase, normalize_code(code)
    return None


def is_likely_continuation(current: dict | None, cluster: dict, max_gap: float = 24) -> bool:
    if current is None or "_Last Top" not in current:
        return False
    gap = cluster["top"] - current["_Last Top"]
    return 0 < gap <= max_gap


def remember_cluster_top(current: dict, cluster: dict):
    current["_Last Top"] = cluster["top"]


def resolve_dados_dir(ref: Path = DADOS_REF) -> Path:
    ref = Path(ref)
    if ref.is_dir() or ref.is_symlink():
        return ref.resolve()
    if ref.exists():
        command = [
            "osascript",
            "-e", f'tell application "Finder" to set targetItem to (original item of (POSIX file "{ref}" as alias))',
            "-e", "POSIX path of (targetItem as alias)",
        ]
        result = subprocess.run(command, check=True, capture_output=True, text=True)
        target = Path(result.stdout.strip()).resolve()
        if target.exists():
            return target
    raise FileNotFoundError(f"Não foi possível resolver {ref} para um diretório de dados.")


def score_spreadsheet(path: Path, dados_dir: Path) -> tuple:
    name = path.name.lower()
    distance = len(path.parts)
    return (
        4 if path.parent == dados_dir else 0,
        3 if "turmasdsc" in name else 0,
        2 if "dsc" in name else 0,
        1 if path.suffix.lower() == ".numbers" else 0,
        -distance,
        path.stat().st_mtime,
    )


def find_spreadsheet(dados_dir: Path) -> Path:
    candidates = []
    seen = set()
    search_roots = [dados_dir, dados_dir.parent, dados_dir.parent.parent]
    for root in search_roots:
        if not root.exists():
            continue
        for pattern in ("*.numbers", "*.xlsx", "*.csv"):
            for path in list(root.glob(pattern)) + list(root.rglob(pattern)):
                if path not in seen:
                    seen.add(path)
                    candidates.append(path)
    if not candidates:
        raise FileNotFoundError("Nenhuma planilha .numbers, .xlsx ou .csv foi localizada a partir de Dados/.")
    return sorted(candidates, key=lambda path: score_spreadsheet(path, dados_dir), reverse=True)[0]


def read_numbers_workbook(path: Path) -> pd.DataFrame:
    document = Document(str(path))
    best_header = None
    best_rows = None
    best_score = -1
    for sheet in document.sheets:
        for table in sheet.tables:
            rows = list(table.rows(values_only=True))
            non_empty = [row for row in rows if any(value not in (None, "") for value in row)]
            if len(non_empty) < 2:
                continue
            header = [normalize_space(value) for value in non_empty[0]]
            score = sum(
                1
                for item in header
                if any(token in strip_accents(item).lower() for token in ["codigo", "curso", "professor", "fase", "grupo"])
            )
            if score > best_score:
                best_header = header
                best_rows = non_empty[1:]
                best_score = score
    if best_header is None:
        raise ValueError(f"Nenhuma tabela utilizável foi encontrada em {path}.")
    return pd.DataFrame(best_rows, columns=best_header)


def read_spreadsheet(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".numbers":
        return read_numbers_workbook(path)
    if suffix == ".xlsx":
        return pd.read_excel(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Formato de planilha não suportado: {path}")


def normalize_planilha(raw_df: pd.DataFrame, fonte: str) -> pd.DataFrame:
    col_map = {strip_accents(str(col)).lower().replace(".", "").replace("!", "").strip(): col for col in raw_df.columns}

    def pick(*names):
        for name in names:
            key = strip_accents(name).lower().replace(".", "").replace("!", "").strip()
            if key in col_map:
                return col_map[key]
        return None

    nome_col = pick("Nome", "Turma", "Disciplina")
    codigo_col = pick("Código", "Codigo", "Cod", "Cód")
    curso_col = pick("Curso")
    professor_col = pick("Professor")
    fase_col = pick("Fase")
    grupo_col = pick("Grupo")
    espaco_col = pick("Espaço", "Espacos", "Espaços", "Espaco")
    laboratorio_col = pick("Laboratório", "Laboratorio", "Lab", "Lab.")
    dsc_col = pick("!DSC", "!_DSC", "Não DSC", "Nao DSC", "DSC")
    concentrado_col = pick("Concentrado", "Conc", "Conc.")
    ead_col = pick("EAD")
    teoria_col = pick("Créditos Teóricos", "Creditos Teoricos", "Crédito Teórico", "Credito Teorico")
    pratica_col = pick("Créditos Práticos", "Creditos Praticos", "Crédito Prático", "Credito Pratico")
    total_col = pick("Créditos", "Creditos", "Crédito", "Credito", "Cre", "Cre.", "Créd", "Créd.")
    horario_col = pick("Horário", "Horario")
    weekdays = [("Seg", pick("Seg")), ("Ter", pick("Ter")), ("Qua", pick("Qua")), ("Qui", pick("Qui")), ("Sex", pick("Sex")), ("Sab", pick("Sab"))]

    missing = []
    for label, column in {
        "Código": codigo_col,
        "Nome": nome_col,
        "Curso": curso_col,
        "Professor": professor_col,
        "Espaço/Espaços": espaco_col,
        "Fase": fase_col,
        "Grupo": grupo_col,
        "Laboratório": laboratorio_col,
        "!DSC": dsc_col,
        "Concentrado": concentrado_col,
        "EAD": ead_col,
    }.items():
        if column is None:
            missing.append(label)
    if teoria_col is None and pratica_col is None and total_col is None:
        missing.append("Créditos Teóricos/Práticos ou Créditos")
    if horario_col is None and not any(column for _, column in weekdays):
        missing.append("Horário ou colunas Seg/Ter/Qua/Qui/Sex/Sab")
    if missing:
        raise KeyError(f"Colunas obrigatórias não encontradas na planilha {fonte}: {', '.join(missing)}")

    def tokens_from_horario(value):
        text = normalize_space(value)
        if not text:
            return []
        tokens = []
        current_day = ""
        for part in re.split(r"[;,]\s*|\s+", text):
            part = normalize_space(part)
            if not part:
                continue
            day = strip_accents(part[:3]).title()
            if day in DAY_ORDER:
                current_day = day
                remainder = part[3:].strip()
                if not remainder:
                    continue
                part = remainder
            token = normalize_time_token(part)
            if current_day and re.fullmatch(r"\d{1,2}/\d{1,2}C?", token):
                tokens.append(f"{current_day}:{token}")
        return tokens

    rows = []
    for _, row in raw_df.iterrows():
        nome = clean_placeholder(row.get(nome_col))
        curso = clean_placeholder(row.get(curso_col))
        if not nome and not curso:
            continue

        horario_tokens = []
        if horario_col:
            horario_tokens.extend(tokens_from_horario(row.get(horario_col)))
        for dia, col in weekdays:
            if not col:
                continue
            value = clean_placeholder(row.get(col))
            if value:
                for token in re.split(r"\s+", value):
                    token = normalize_time_token(token)
                    if re.fullmatch(r"\d{1,2}/\d{1,2}C?", token):
                        horario_tokens.append(f"{dia}:{token}")

        credito_teorico = row.get(teoria_col)
        credito_pratico = row.get(pratica_col)
        credito_total = row.get(total_col) if total_col else None

        teo = int(float(credito_teorico)) if normalize_space(credito_teorico) else None
        pra = int(float(credito_pratico)) if normalize_space(credito_pratico) else None
        total = int(float(credito_total)) if normalize_space(credito_total) else (teo or 0) + (pra or 0)
        if teo is None and pra is None:
            teo, pra = total, 0

        rows.append({
            "Fonte": fonte,
            "Código": normalize_code(row.get(codigo_col)),
            "Nome": nome,
            "Curso": curso,
            "Professor": clean_placeholder(row.get(professor_col)),
            "Horário Tokens": sort_tokens(horario_tokens),
            "Horário": compose_horario(horario_tokens),
            "Fase": normalize_space(row.get(fase_col)),
            "Grupo": normalize_space(row.get(grupo_col)),
            "Espaço": clean_placeholder(row.get(espaco_col)),
            "Laboratório": normalize_bool(row.get(laboratorio_col)),
            "!DSC": normalize_bool(row.get(dsc_col)),
            "Concentrado": normalize_bool(row.get(concentrado_col)),
            "EAD": normalize_bool(row.get(ead_col)),
            "Créditos Teóricos": teo,
            "Créditos Práticos": pra,
            "Total de Créditos": total,
        })

    df = pd.DataFrame(rows)
    return add_normalized_keys(df)


def parse_course_pdf(path: Path, curso: str) -> pd.DataFrame:
    records = []
    current = None
    with pdfplumber.open(path) as pdf:
        stop = False
        for page in pdf.pages:
            if stop:
                break
            current = None
            for cluster in cluster_page_words(page):
                words = cluster["words"]
                line = " ".join(word["text"] for word in words)
                if "Período do Regime Concentrado" in line:
                    stop = True
                    break

                phase_code = extract_course_phase_code(words)
                if phase_code:
                    phase, code = phase_code
                    title_words = [word["text"] for word in words if 100 <= word["x0"] < 340]
                    title_text = " ".join(title_words).strip()
                    name, professor = extract_name_professor(title_words)
                    professor_open = has_open_professor_parenthesis(title_text)
                    credit_words = [word["text"] for word in words if 365 <= word["x0"] < 383 and re.fullmatch(r"\d+", word["text"])]
                    credits = int(credit_words[0]) if credit_words else 0
                    horario_tokens = []
                    for word in words:
                        if 380 <= word["x0"] < 565 and re.fullmatch(r"\d{1,2}/\d{1,2}|C", word["text"]):
                            token = word["text"]
                            if token == "C" and horario_tokens:
                                if not horario_tokens[-1].endswith("C"):
                                    horario_tokens[-1] = f"{horario_tokens[-1]}C"
                            elif token != "0":
                                horario_tokens.append(f"{assign_day(word['x0'], COURSE_DAY_X)}:{normalize_time_token(token)}")
                    current = {
                        "Fonte": path.name,
                        "Código": normalize_code(code),
                        "Nome": name,
                        "Curso": curso,
                        "Professor": professor,
                        "Fase": phase,
                        "Grupo": "",
                        "Espaço": "",
                        "Laboratório": False,
                        "!DSC": False,
                        "Concentrado": False,
                        "Créditos Teóricos": credits,
                        "Créditos Práticos": 0,
                        "Total de Créditos": credits,
                        "Horário Tokens": sort_tokens(horario_tokens),
                        "_Professor Aberto": professor_open,
                        "_Last Top": cluster["top"],
                    }
                    records.append(current)
                elif current:
                    continuation = " ".join(word["text"] for word in words if 100 <= word["x0"] < 340).strip()
                    if current.get("_Professor Aberto") and is_likely_continuation(current, cluster):
                        if continuation:
                            professor_part = continuation.removesuffix(")").strip()
                            current["Professor"] = normalize_space(f"{current['Professor']} {professor_part}")
                            remember_cluster_top(current, cluster)
                            if ")" in continuation:
                                current["_Professor Aberto"] = False
                    elif continuation and is_likely_continuation(current, cluster) and not re.search(r"[A-Z]{3}\.\d{4}", continuation):
                        current["Nome"] = normalize_space(f"{current['Nome']} {continuation}")
                        remember_cluster_top(current, cluster)

                    extra_words = [word for word in words if 380 <= word["x0"] < 565 and re.fullmatch(r"\d{1,2}/\d{1,2}|C", word["text"])]
                    if extra_words:
                        tokens = list(current["Horário Tokens"])
                        for word in extra_words:
                            token = word["text"]
                            if token == "C" and tokens:
                                if not tokens[-1].endswith("C"):
                                    tokens[-1] = f"{tokens[-1]}C"
                            else:
                                tokens.append(f"{assign_day(word['x0'], COURSE_DAY_X)}:{normalize_time_token(token)}")
                        current["Horário Tokens"] = sort_tokens(tokens)

    if not records:
        raise ValueError(f"Nenhuma turma foi extraída de {path}.")

    df = pd.DataFrame(records)
    df["Horário"] = df["Horário Tokens"].apply(compose_horario)
    df = df.drop(columns=[column for column in df.columns if column.startswith("_")], errors="ignore")
    return deduplicate_pdf_rows(add_normalized_keys(df))


def parse_dsc_pdf(path: Path) -> pd.DataFrame:
    records = []
    current = None
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            current = None
            for cluster in cluster_page_words(page):
                words = cluster["words"]
                code_parts = [word["text"] for word in words if word["x0"] < 120]
                code_text = " ".join(code_parts)
                has_code = re.search(r"[A-Z]{3}\.", code_text)

                if not has_code:
                    if is_likely_continuation(current, cluster):
                        changed = False
                        continuation_parts = [word["text"] for word in words if 120 <= word["x0"] < 310]
                        continuation = normalize_space(" ".join(continuation_parts))
                        if continuation and not re.search(r"[A-Z]{3}\.\d{4}", continuation):
                            current["Nome"] = normalize_space(f"{current['Nome']} {continuation}")
                            changed = True

                        extra_tokens = list(current["Horário Tokens"])
                        for word in words:
                            if 520 <= word["x0"] < 705 and re.fullmatch(r"\d{1,2}/\d{1,2}|C", word["text"]):
                                token = word["text"]
                                if token == "C" and extra_tokens:
                                    if not extra_tokens[-1].endswith("C"):
                                        extra_tokens[-1] = f"{extra_tokens[-1]}C"
                                        changed = True
                                else:
                                    extra_tokens.append(f"{assign_day(word['x0'], DSC_DAY_X)}:{normalize_time_token(token)}")
                                    changed = True
                        if changed:
                            current["Horário Tokens"] = sort_tokens(extra_tokens)
                            current["Horário"] = compose_horario(current["Horário Tokens"])
                            remember_cluster_top(current, cluster)
                    continue

                code = normalize_code(code_text)
                if not re.fullmatch(r"[A-Z]{3}\.\d{4}\.\d{2}\.\d{3}-\d", code):
                    continue

                name_parts = [word["text"] for word in words if 120 <= word["x0"] < 310]
                course_parts = [word["text"] for word in words if 720 <= word["x0"] < 760]
                if not name_parts or not course_parts:
                    current = None
                    continue

                professor_parts = [word["text"] for word in words if 310 <= word["x0"] < 470]
                credit_parts = [word["text"] for word in words if 470 <= word["x0"] < 520 and re.fullmatch(r"\d+", word["text"])]
                horario_tokens = []
                for word in words:
                    if 520 <= word["x0"] < 705 and re.fullmatch(r"\d{1,2}/\d{1,2}|C", word["text"]):
                        token = word["text"]
                        if token == "C" and horario_tokens:
                            if not horario_tokens[-1].endswith("C"):
                                horario_tokens[-1] = f"{horario_tokens[-1]}C"
                        else:
                            horario_tokens.append(f"{assign_day(word['x0'], DSC_DAY_X)}:{normalize_time_token(token)}")

                teo = int(credit_parts[0]) if len(credit_parts) > 0 else 0
                pra = int(credit_parts[1]) if len(credit_parts) > 1 else 0
                professor = re.sub(r"^\d+\s*", "", " ".join(professor_parts).strip())

                current = {
                    "Fonte": path.name,
                    "Código": code,
                    "Nome": " ".join(name_parts).strip(),
                    "Curso": " ".join(course_parts).strip(),
                    "Professor": professor,
                    "Horário Tokens": sort_tokens(horario_tokens),
                    "Horário": compose_horario(horario_tokens),
                    "Fase": normalize_space(next((word["text"] for word in words if 760 <= word["x0"] < 780 and re.fullmatch(r"\d+", word["text"])), "")),
                    "Grupo": normalize_space(next((word["text"] for word in words if 780 <= word["x0"] < 800), "")),
                    "Espaço": "",
                    "Laboratório": False,
                    "!DSC": False,
                    "Concentrado": False,
                    "Créditos Teóricos": teo,
                    "Créditos Práticos": pra,
                    "Total de Créditos": teo + pra,
                    "_Last Top": cluster["top"],
                }
                records.append(current)

    if not records:
        raise ValueError(f"Nenhuma turma foi extraída de {path}.")

    df = pd.DataFrame(records)
    df = df.drop(columns=[column for column in df.columns if column.startswith("_")], errors="ignore")
    return deduplicate_pdf_rows(add_normalized_keys(df))


def horario_sem_concentrado_key(tokens) -> tuple:
    normalized = []
    for item in tokens if isinstance(tokens, tuple) else tuple():
        if ":" in item:
            day, token = item.split(":", 1)
            normalized.append(f"{day}:{token.removesuffix('C')}")
    return sort_tokens(normalized)


def add_normalized_keys(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Código Key"] = out["Código"].map(key_text)
    out["Nome Key"] = out["Nome"].map(key_text)
    out["Curso Key"] = out["Curso"].map(key_text)
    out["Professor Key"] = out["Professor"].map(key_text)
    out["Horário Key"] = out["Horário Tokens"].map(lambda tokens: tuple(tokens) if isinstance(tokens, tuple) else tuple())
    out["Horário Sem C Key"] = out["Horário Tokens"].map(horario_sem_concentrado_key)
    return out


def deduplicate_pdf_rows(df: pd.DataFrame) -> pd.DataFrame:
    group_cols = ["Fonte", "Código Key", "Curso Key"]
    rows = []
    for _, group in df.groupby(group_cols, dropna=False, sort=False):
        base = group.iloc[0].copy()
        for column in ["Nome", "Professor", "Fase", "Grupo"]:
            values = [value for value in group[column].tolist() if normalize_space(value)]
            if values:
                base[column] = max(values, key=len)
        base["Créditos Teóricos"] = int(group["Créditos Teóricos"].max())
        base["Créditos Práticos"] = int(group["Créditos Práticos"].max())
        base["Total de Créditos"] = int(group["Total de Créditos"].max())
        merged_tokens = []
        for tokens in group["Horário Tokens"]:
            merged_tokens.extend(list(tokens))
        base["Horário Tokens"] = sort_tokens(merged_tokens)
        base["Horário"] = compose_horario(base["Horário Tokens"])
        rows.append(base)
    return add_normalized_keys(pd.DataFrame(rows)).reset_index(drop=True)


def display_result(title: str, df: pd.DataFrame, columns: list[str] | None = None):
    display(Markdown(f"### {title}"))
    if df.empty:
        display(Markdown("Nenhuma inconsistência encontrada."))
        return
    if columns:
        display(df.loc[:, columns])
    else:
        display(df)


def inconsistent_credit_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df[df["Total de Créditos"].fillna(0).astype(int) % 2 == 1].copy()
    out = out[["Fonte", "Código", "Nome", "Curso", "Créditos Teóricos", "Créditos Práticos", "Total de Créditos"]]
    out = out.rename(columns={
        "Créditos Teóricos": "Teóricos",
        "Créditos Práticos": "Práticos",
        "Total de Créditos": "Total",
    })
    return out.sort_values(["Fonte", "Código", "Nome"])


def has_concentrated_time(df: pd.DataFrame) -> pd.Series:
    return df["Horário Tokens"].map(lambda tokens: any(token.endswith("C") for token in tokens))


def has_ead_name(df: pd.DataFrame) -> pd.Series:
    return df["Nome Key"].str.contains(r"\(EAD\)", regex=True, na=False)


def concentrated_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df[has_concentrated_time(df) & ~has_ead_name(df)].copy()
    return out[["Fonte", "Código", "Nome", "Curso", "Professor", "Horário"]].sort_values(["Fonte", "Código", "Nome"])


def ead_concentrated_rows(planilha_df: pd.DataFrame, fontes_df: pd.DataFrame) -> pd.DataFrame:
    ead_found = fontes_df[has_concentrated_time(fontes_df) & has_ead_name(fontes_df)].copy()
    ead_keys = set(zip(ead_found["Código Key"], ead_found["Curso Key"]))
    source_map = ead_found.groupby(["Código Key", "Curso Key"], dropna=False)["Fonte"].apply(lambda values: ", ".join(sorted(set(values)))).to_dict()

    mask = planilha_df.apply(lambda row: (row["Código Key"], row["Curso Key"]) in ead_keys, axis=1) & ~planilha_df["EAD"]
    out = planilha_df[mask].copy()
    out["Fonte EAD"] = out.apply(lambda row: source_map.get((row["Código Key"], row["Curso Key"]), ""), axis=1)
    out["EAD marcado"] = out["EAD"]
    out["Tipo de inconsistência"] = "EAD não marcado na planilha"
    return out[["Fonte EAD", "Código", "Nome", "Curso", "Professor", "Horário", "EAD marcado", "Tipo de inconsistência"]].sort_values(["Fonte EAD", "Código", "Nome"])


def rows_without_professor(df: pd.DataFrame) -> pd.DataFrame:
    out = df[(df["Professor Key"] == "") & ~df["Código Key"].str.startswith("PDE.", na=False)].copy()
    if out.empty:
        return pd.DataFrame(columns=["Código", "Nome", "Fonte"])
    grouped = (
        out.groupby(["Código Key", "Nome Key"], dropna=False, sort=False)
        .agg({
            "Código": "first",
            "Nome": "first",
            "Fonte": lambda values: ", ".join(sorted(set(values))),
        })
        .reset_index(drop=True)
    )
    return grouped[["Código", "Nome", "Fonte"]].sort_values(["Código", "Nome"]).reset_index(drop=True)


def parse_time_range_token(token: str) -> tuple[int, int] | None:
    clean_token = normalize_time_token(token).removesuffix("C")
    match = re.fullmatch(r"(\d{1,2})/(\d{1,2})", clean_token)
    if not match:
        return None
    start, end = map(int, match.groups())
    if end < start:
        return None
    return start, end


def count_periods_in_time_token(token: str) -> int | None:
    parsed = parse_time_range_token(token)
    if parsed is None:
        return None
    start, end = parsed
    return end - start + 1


def count_schedule_periods(tokens) -> int:
    total = 0
    for item in tokens if isinstance(tokens, tuple) else tuple():
        _, token = item.split(":", 1) if ":" in item else ("", item)
        periods = count_periods_in_time_token(token)
        if periods is not None:
            total += periods
    return total


def schedule_credit_mismatch_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Períodos no Horário"] = out["Horário Tokens"].map(count_schedule_periods)
    expected = out["Créditos Teóricos"].fillna(0).astype(int) + out["Créditos Práticos"].fillna(0).astype(int)
    mask = (expected > 0) & (out["Períodos no Horário"] != expected) & ~out["Código Key"].str.startswith("PDE.", na=False)
    out = out[mask].copy()
    if out.empty:
        return pd.DataFrame(columns=["Código", "Nome", "Horário informado", "Créditos", "Horário", "Fonte"])
    out = out.rename(columns={
        "Horário": "Horário informado",
        "Total de Créditos": "Créditos",
        "Períodos no Horário": "Horário",
    })
    grouped = (
        out.groupby(["Código Key", "Nome Key", "Horário Key", "Créditos", "Horário"], dropna=False, sort=False)
        .agg({
            "Código": "first",
            "Nome": "first",
            "Horário informado": "first",
            "Créditos": "first",
            "Horário": "first",
            "Fonte": lambda values: ", ".join(sorted(set(values))),
        })
        .reset_index(drop=True)
    )
    return grouped[["Código", "Nome", "Horário informado", "Créditos", "Horário", "Fonte"]].sort_values(["Código", "Nome", "Horário informado"]).reset_index(drop=True)


def zero_schedule_credit_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df[df["Horário"] == 0].copy()
    out = out.drop(columns=["Horário", "Horário informado", "Créditos"], errors="ignore")
    return out.reset_index(drop=True)


def nonzero_schedule_credit_mismatch_rows(df: pd.DataFrame) -> pd.DataFrame:
    out = df[df["Horário"] != 0].copy()
    return out.reset_index(drop=True)


def explode_schedule(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        tokens = row.get("Horário Tokens")
        for item in tokens if isinstance(tokens, tuple) else tuple():
            if ":" not in item:
                continue
            day, token = item.split(":", 1)
            parsed = parse_time_range_token(token)
            if parsed is None:
                continue
            start, end = parsed
            for period in range(start, end + 1):
                expanded = row.copy()
                expanded["Dia"] = day
                expanded["Período"] = period
                expanded["Horário de Conflito"] = f"{day} {period}"
                rows.append(expanded)
    if not rows:
        return pd.DataFrame(columns=list(df.columns) + ["Dia", "Período", "Horário de Conflito"])
    return pd.DataFrame(rows)


def find_conflicts(df: pd.DataFrame, group_fields: list[str], label_map: dict[str, str], type_label: str) -> pd.DataFrame:
    exploded = explode_schedule(df)
    for field in group_fields:
        exploded = exploded[exploded[field].astype(str).str.strip() != ""]
    grouped = exploded.groupby(group_fields + ["Dia", "Período"], dropna=False)
    out = grouped.filter(lambda group: len(group) > 1).copy()
    base_columns = list(label_map.values()) + ["Horário", "Código", "Nome", "Curso", "Tipo de conflito"]
    unique_columns = []
    for column in base_columns:
        if column not in unique_columns:
            unique_columns.append(column)
    if out.empty:
        return pd.DataFrame(columns=unique_columns)
    out["Tipo de conflito"] = type_label
    out = out.rename(columns=label_map)
    keep = []
    for column in list(label_map.values()) + ["Horário de Conflito", "Código", "Nome", "Curso", "Tipo de conflito"]:
        if column not in keep:
            keep.append(column)
    out = out[keep].drop_duplicates()
    out = out.rename(columns={"Horário de Conflito": "Horário"})
    sort_columns = [column for column in out.columns if column != "Tipo de conflito"]
    return out.sort_values(sort_columns).reset_index(drop=True)


def normalize_lab_space(value: str) -> str:
    text = key_text(value).replace("LAB_", "")
    return text


def lab_conflicts(planilha_df: pd.DataFrame) -> pd.DataFrame:
    valid_spaces = LABORATORIOS_VALIDOS | {space.replace("LAB_", "") for space in LABORATORIOS_VALIDOS}
    out = planilha_df[planilha_df["Espaço"].map(normalize_lab_space).isin(valid_spaces) & ~planilha_df["Laboratório"]].copy()
    out["Tipo de inconsistência"] = "Laboratório não marcado"
    return out[["Fonte", "Código", "Nome", "Curso", "Professor", "Horário", "Espaço", "Tipo de inconsistência"]].sort_values(["Código", "Nome"])


def dsc_flag_conflicts(planilha_df: pd.DataFrame, course_pdf_df: pd.DataFrame) -> pd.DataFrame:
    course_keys = set(zip(course_pdf_df["Código Key"], course_pdf_df["Nome Key"]))
    mask = ~planilha_df.apply(lambda row: (row["Código Key"], row["Nome Key"]) in course_keys, axis=1) & ~planilha_df["!DSC"]
    out = planilha_df[mask].copy()
    out["Tipo de inconsistência"] = "!DSC não marcado"
    return out[["Fonte", "Código", "Nome", "Curso", "Professor", "Horário", "Tipo de inconsistência"]].sort_values(["Curso", "Nome"])


def concentrado_flag_conflicts(planilha_df: pd.DataFrame) -> pd.DataFrame:
    mask = has_concentrated_time(planilha_df) & ~has_ead_name(planilha_df) & ~planilha_df["Concentrado"]
    out = planilha_df[mask].copy()
    out["Concentrado marcado"] = out["Concentrado"]
    out["Tipo de inconsistência"] = "Concentrado não marcado na planilha"
    return out[["Fonte", "Código", "Nome", "Curso", "Professor", "Horário", "Concentrado marcado", "Tipo de inconsistência"]].sort_values(["Curso", "Nome"])


def compare_missing_classes(pdf_df: pd.DataFrame, planilha_df: pd.DataFrame, compare_course: bool) -> pd.DataFrame:
    if compare_course:
        planilha_keys = set(zip(planilha_df["Curso Key"], planilha_df["Código Key"], planilha_df["Nome Key"]))
        mask = ~pdf_df.apply(lambda row: (row["Curso Key"], row["Código Key"], row["Nome Key"]) in planilha_keys, axis=1)
    else:
        planilha_keys = set(zip(planilha_df["Código Key"], planilha_df["Nome Key"]))
        mask = ~pdf_df.apply(lambda row: (row["Código Key"], row["Nome Key"]) in planilha_keys, axis=1)
    out = pdf_df[mask].copy()
    return out[["Fonte", "Código", "Nome", "Curso", "Professor", "Horário"]].sort_values(["Fonte", "Curso", "Código", "Nome"])


def compare_planilha_vs_pdfs(planilha_df: pd.DataFrame, pdf_df: pd.DataFrame):
    merged = planilha_df.merge(
        pdf_df,
        on="Código Key",
        how="inner",
        suffixes=(" Planilha", " PDF"),
    )
    merged = merged[merged["Código Key"] != ""]

    def build_difference(mask, label):
        out = merged[mask].copy()
        out["Código"] = out["Código Planilha"]
        out["Fonte PDF"] = out["Fonte PDF"]
        out["Tipo de inconsistência"] = label
        return out[[
            "Código",
            "Nome Planilha", "Nome PDF",
            "Professor Planilha", "Professor PDF",
            "Horário Planilha", "Horário PDF",
            "Curso Planilha", "Curso PDF",
            "Fonte PDF",
            "Tipo de inconsistência",
        ]].drop_duplicates().sort_values(["Fonte PDF", "Código", "Tipo de inconsistência"])

    nome_diferente = build_difference(merged["Nome Planilha"] != merged["Nome PDF"], "Nome diferente")
    professor_diferente = build_difference(merged["Professor Planilha"] != merged["Professor PDF"], "Professor diferente")
    horario_diferente = build_difference(merged["Horário Sem C Key Planilha"] != merged["Horário Sem C Key PDF"], "Horário diferente")
    curso_diferente = build_difference(merged["Curso Key Planilha"] != merged["Curso Key PDF"], "Curso diferente")
    return nome_diferente, professor_diferente, horario_diferente, curso_diferente


def dataframe_to_markdown(df: pd.DataFrame, columns: list[str] | None = None) -> str:
    table = df if columns is None else df.loc[:, columns]
    if table.empty:
        return "Nenhuma inconsistência encontrada."
    return table.to_markdown(index=False)


def markdown_section(title: str, df: pd.DataFrame, columns: list[str] | None = None) -> str:
    return f"{title}\n\n{dataframe_to_markdown(df, columns)}\n"

In [3]:
dados_dir = resolve_dados_dir()
spreadsheet_path = find_spreadsheet(dados_dir)
planilha_raw = read_spreadsheet(spreadsheet_path)
planilha_df = normalize_planilha(planilha_raw, spreadsheet_path.name)

print(f"Diretório de dados resolvido: {dados_dir}")
print(f"Planilha selecionada: {spreadsheet_path.name}")
print(f"Linhas normalizadas da planilha: {len(planilha_df)}")

display_result(
    "Planilha normalizada - amostra",
    planilha_df.head(10),
    ["Fonte", "Código", "Nome", "Curso", "Professor", "Horário", "Fase", "Grupo", "Espaço", "Laboratório", "!DSC", "Concentrado", "EAD", "Total de Créditos"],
)

Diretório de dados resolvido: /Users/daltonreis/Library/Mobile Documents/com~apple~Numbers/Documents/DSC/2026_2/2026-06-04
Planilha selecionada: TurmasDSC_2026_2.numbers
Linhas normalizadas da planilha: 148


### Planilha normalizada - amostra

,Fonte,Código,Nome,Curso,Professor,Horário,Fase,Grupo,Espaço,Laboratório,!DSC,Concentrado,EAD,Total de Créditos
0,TurmasDSC_2026_2.numbers,SIS.0119.00.001-9,Optativa,SIS-N,,,8.0,A,,False,False,False,False,4
1,TurmasDSC_2026_2.numbers,CMP.0175.01.002-7,Trabalho de Conclusão de Curso I,BCC-M,_OfertarSoNoite_,,8.0,A,remota,False,False,False,False,4
2,TurmasDSC_2026_2.numbers,EDU.0542.00.002-1,"Universidade, Ciência e Pesquisa (EAD)",BCC-N,Adolfo Ramos Lamar,Sex 12/13,1.0,A,,False,True,False,True,2
3,TurmasDSC_2026_2.numbers,SIS.0102.00.001-0,Banco de Dados,BCC-N,Alexander Roberto Valdameri,Qui 14/15,2.0,A,S-401,True,False,False,False,6
4,TurmasDSC_2026_2.numbers,SIS.0102.00.003-7,Banco de Dados,BCC-N,Alexander Roberto Valdameri,Qui 12/13,2.0,B,S-401,True,False,False,False,6
5,TurmasDSC_2026_2.numbers,SIS.0125.00.001-7,Banco de Dados I,BCD-N,Alexander Roberto Valdameri,Seg 12/15,2.0,A,S-401,True,False,False,False,6
6,TurmasDSC_2026_2.numbers,SIS.0125.00.002-5,Banco de Dados I,BCD-N,Alexander Roberto Valdameri,Seg 12/15,2.0,B,S-401,True,False,False,False,6
7,TurmasDSC_2026_2.numbers,SIS.0114.02.001-1,Banco de Dados II,SIS-N,Alexander Roberto Valdameri,Qua 12/15,4.0,A,S-224,True,False,False,False,4
8,TurmasDSC_2026_2.numbers,CMP.0168.00.001-1,Programação Orientada a Objetos,BCC-N,André Felipe Bürger,Seg 12/15; Qui 12/13,2.0,A,S-415,True,False,False,False,6
9,TurmasDSC_2026_2.numbers,CMP.0168.00.004-6,Programação Orientada a Objetos,BCC-N,André Felipe Bürger,Ter 12/15; Qui 14/15,2.0,B,S-415,True,False,False,False,6


In [4]:
course_pdf_frames = [parse_course_pdf(dados_dir / filename, course) for filename, course in COURSE_PDFS.items()]
course_pdf_df = pd.concat(course_pdf_frames, ignore_index=True)
dsc_pdf_df = parse_dsc_pdf(dados_dir / "DSC.pdf")
all_pdf_df = pd.concat([course_pdf_df, dsc_pdf_df], ignore_index=True)

pdf_summary = pd.concat([
    course_pdf_df.groupby("Fonte", dropna=False).size().rename("Turmas"),
    dsc_pdf_df.groupby("Fonte", dropna=False).size().rename("Turmas"),
]).reset_index()

print(f"Turmas extraídas dos PDFs de curso: {len(course_pdf_df)}")
print(f"Turmas extraídas do DSC.pdf: {len(dsc_pdf_df)}")

display_result("Resumo da leitura dos PDFs", pdf_summary)
display_result(
    "Amostra consolidada dos PDFs",
    all_pdf_df.head(12),
    ["Fonte", "Código", "Nome", "Curso", "Professor", "Horário", "Fase", "Grupo", "Total de Créditos"],
)

Turmas extraídas dos PDFs de curso: 138
Turmas extraídas do DSC.pdf: 100


### Resumo da leitura dos PDFs

,Fonte,Turmas
0,BCC_mat.pdf,20
1,BCC_not.pdf,81
2,BCD_not.pdf,14
3,SIS_not.pdf,23
4,DSC.pdf,100


### Amostra consolidada dos PDFs

,Fonte,Código,Nome,Curso,Professor,Horário,Fase,Grupo,Total de Créditos
0,BCC_mat.pdf,CMP.0167.02.002-7,Arquitetura de Computadores II,BCC-M,Miguel Alexandre Wisintainer,Seg 1/4,2,,4
1,BCC_mat.pdf,CMP.0168.00.002-0,Programação Orientada a Objetos,BCC-M,Luciana Pereira de Araújo Kohler,Qua 1/4; Qui 1/2,2,,7
2,BCC_mat.pdf,CMP.0169.00.002-3,Lógica para Computação,BCC-M,,Sex 1/4,2,,4
3,BCC_mat.pdf,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,BCC-M,,,2,,2
4,BCC_mat.pdf,SIS.0102.00.002-9,Banco de Dados,BCC-M,Luciana Pereira de Araújo Kohler,Qui 3/4,2,,6
5,BCC_mat.pdf,CMP.0170.02.002-4,Programação Web II,BCC-M,Luciana Pereira de Araújo Kohler,Seg 1/4,4,,5
6,BCC_mat.pdf,CMP.0178.00.002-2,Teoria dos Grafos,BCC-M,Patricia Kayser Vargas Mangan,Qui 1/4,4,,4
7,BCC_mat.pdf,CMP.0183.00.002-7,Compiladores,BCC-M,Joyce Martins,Qua 1/4,4,,4
8,BCC_mat.pdf,HIS.0116.00.003-2,História da Cultura Afro-brasileira e Indígena (EAD),BCC-M,Ricardo Duwe,Sex 1/2C,4,,2
9,BCC_mat.pdf,MAT.0131.00.002-6,Estatística,BCC-M,Luciane Zickuhr Tomelin,Ter 1/4,4,,4


In [5]:
# Preparação das fontes combinadas usadas pelas regras.
# A planilha, os PDFs dos cursos e o DSC.pdf são unidos em uma única tabela para
# permitir validações que precisam enxergar todas as fontes ao mesmo tempo.
fontes_completas_df = pd.concat([planilha_df, course_pdf_df, dsc_pdf_df], ignore_index=True)


In [6]:
# Regra A1: lista turmas que aparecem nos PDFs dos cursos, mas não aparecem na planilha.
# A comparação usa Curso + Código + Nome, porque cada PDF de curso representa uma oferta
# vinculada a um curso específico, como BCC-N, BCC-M, BCD-N ou SIS-N.
ausentes_cursos = compare_missing_classes(course_pdf_df, planilha_df, compare_course=True)
display_result("A1) Turmas dos cursos PDFs ausentes na planilha", ausentes_cursos)


### A1) Turmas dos cursos PDFs ausentes na planilha

Nenhuma inconsistência encontrada.

In [7]:
# Regra A2: lista turmas que aparecem no DSC.pdf, mas não aparecem na planilha.
# A comparação usa Código + Nome, sem exigir Curso, porque o DSC.pdf é a fonte geral
# do departamento e pode reunir turmas de cursos diferentes.
ausentes_dsc = compare_missing_classes(dsc_pdf_df, planilha_df, compare_course=False)
display_result("A2) Turmas do DSC ausentes na planilha", ausentes_dsc)


### A2) Turmas do DSC ausentes na planilha

Nenhuma inconsistência encontrada.

In [8]:
# Regra A3: encontra diferenças de horário entre a planilha e os PDFs para o mesmo código.
# Esta célula também calcula as demais diferenças entre planilha e PDFs, pois todas usam
# o mesmo cruzamento por código; na comparação de horário, Sex 1/2 e Sex 1/2C são iguais.
nome_diferente, professor_diferente, horario_diferente, curso_diferente = compare_planilha_vs_pdfs(planilha_df, all_pdf_df)
professor_diferente = professor_diferente[
    (professor_diferente["Professor Planilha"].map(normalize_space) != "")
    & (professor_diferente["Professor PDF"].map(normalize_space) != "")
].copy()
professor_diferente = professor_diferente.drop(columns=["Tipo de inconsistência"], errors="ignore")
horario_diferente = horario_diferente.drop(columns=["Tipo de inconsistência"], errors="ignore")
curso_diferente = curso_diferente.drop(columns=["Tipo de inconsistência"], errors="ignore")
display_result("A3) Horário diferente", horario_diferente)


### A3) Horário diferente

,Código,Nome Planilha,Nome PDF,Professor Planilha,Professor PDF,Horário Planilha,Horário PDF,Curso Planilha,Curso PDF,Fonte PDF
121,CMP.0172.02.001-3,Eletiva II,Eletiva II,Leandro Werner Ribeiro,Leandro Werner Ribeiro,,Seg 12/15; Ter 12/15; Qua 12/15; Qui 12/15; Sex 12/15C,BCC-N,BCC-N,BCC_not.pdf
135,CMP.0180.00.003-4,Processamento de Linguagem Natural,Processamento de Linguagem Natural,Maiko Rafael Spiess,Maiko Rafael Spiess,Qui 12/15,,BCC-N,BCC-N,BCC_not.pdf
213,SIS.0105.00.001-1,Inovação Tecnológica,Inovação Tecnológica,Simone Erbs da Costa,Simone Erbs da Costa,,Seg 12/15; Ter 12/15; Qua 12/15; Qui 12/15; Sex 12/15; Sab 1/4C,BCC-N,BCC-N,BCC_not.pdf
215,SIS.0105.00.003-8,Inovação Tecnológica,Inovação Tecnológica,Simone Erbs da Costa,Simone Erbs da Costa,,Seg 12/15; Ter 12/15; Qua 12/15; Qui 12/15; Sex 12/15; Sab 1/4C,BCC-N,BCC-N,BCC_not.pdf
140,CDD.0001.00.001-6,Introdução à Ciência de Dados,Introdução à Ciência de Dados,Marcos Antonio Mattedi,Marcos Antonio Mattedi,Seg 12/15,Seg 12/14; Seg 15/15,BCD-N,BCD-N,BCD_not.pdf
106,CDD.0002.00.001-0,Análise Exploratória de Dados,Análise Exploratória de Dados,Jonathan Gil Müller,Jonathan Gil Müller,Ter 12/15,Ter 12/13; Ter 14/15,BCD-N,BCD-N,BCD_not.pdf
108,CDD.0002.00.002-8,Análise Exploratória de Dados,Análise Exploratória de Dados,Jonathan Gil Müller,Jonathan Gil Müller,Qua 12/15,Qua 12/13; Qua 14/15,BCD-N,BCD-N,BCD_not.pdf
5,SIS.0125.00.001-7,Banco de Dados I,Banco de Dados I,Alexander Roberto Valdameri,Alexander Roberto Valdameri,Seg 12/15,Seg 12/13; Seg 14/15,BCD-N,BCD-N,BCD_not.pdf
7,SIS.0125.00.002-5,Banco de Dados I,Banco de Dados I,Alexander Roberto Valdameri,Alexander Roberto Valdameri,Seg 12/15,,BCD-N,BCD-N,BCD_not.pdf
141,CDD.0001.00.001-6,Introdução à Ciência de Dados,Introdução à Ciência de Dados,Marcos Antonio Mattedi,Marcos Antonio Mattedi,Seg 12/15,Seg 12/14; Seg 15/15,BCD-N,BCD-N,DSC.pdf


In [9]:
# Regra A4: encontra diferenças de nome entre a planilha e os PDFs para o mesmo código.
# A comparação usa igualdade direta, preservando diferenças de acento, caixa e espaçamento.
display_result("A4) Nome diferente", nome_diferente)


### A4) Nome diferente

Nenhuma inconsistência encontrada.

In [10]:
# Regra A5: encontra diferenças de curso entre a planilha e os PDFs para o mesmo código.
# A comparação usa o curso normalizado da planilha contra o curso indicado em cada PDF.
display_result("A5) Curso diferente", curso_diferente)


### A5) Curso diferente

,Código,Nome Planilha,Nome PDF,Professor Planilha,Professor PDF,Horário Planilha,Horário PDF,Curso Planilha,Curso PDF,Fonte PDF
238,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,Educação Física - Prática Desportiva II,,,,,BCC-N,BCC-M,BCC_mat.pdf
246,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,Educação Física - Prática Desportiva II,,,,,BCD-N,BCC-M,BCC_mat.pdf
254,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,Educação Física - Prática Desportiva II,,,,,SIS-N,BCC-M,BCC_mat.pdf
232,PDE.0006.00.000-5,Educação Física - Prática Desportiva I,Educação Física - Prática Desportiva I,,,,,BCD-N,BCC-N,BCC_not.pdf
235,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,Educação Física - Prática Desportiva II,,,,,BCC-M,BCC-N,BCC_not.pdf
247,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,Educação Física - Prática Desportiva II,,,,,BCD-N,BCC-N,BCC_not.pdf
255,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,Educação Física - Prática Desportiva II,,,,,SIS-N,BCC-N,BCC_not.pdf
231,PDE.0006.00.000-5,Educação Física - Prática Desportiva I,Educação Física - Prática Desportiva I,,,,,BCC-N,BCD-N,BCD_not.pdf
236,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,Educação Física - Prática Desportiva II,,,,,BCC-M,BCD-N,BCD_not.pdf
240,PDE.0007.00.000-9,Educação Física - Prática Desportiva II,Educação Física - Prática Desportiva II,,,,,BCC-N,BCD-N,BCD_not.pdf


In [11]:
# Regra B1: encontra turmas concentradas não marcadas corretamente na planilha.
# São consideradas concentradas as turmas com C junto ao horário e sem (EAD) no nome;
# o resultado mostra inconsistências entre essa detecção e a coluna Concentrado.
validacao_concentrado = concentrado_flag_conflicts(planilha_df)
display_result("B1) Concentrado não marcado na planilha", validacao_concentrado)


### B1) Concentrado não marcado na planilha

Nenhuma inconsistência encontrada.

In [12]:
# Regra B2: encontra turmas com horário concentrado sem marcação de Concentrado na planilha.
# Esta conferência usa a planilha como fonte e compara a presença de C no horário com
# a coluna Concentrado.
conflitos_flag_concentrado = concentrado_flag_conflicts(planilha_df)
display_result("B2) Conflito de Concentrado", conflitos_flag_concentrado)


### B2) Conflito de Concentrado

Nenhuma inconsistência encontrada.

In [13]:
# Regra B3: encontra turmas em regime EAD não marcadas corretamente na planilha.
# São consideradas EAD as turmas com C junto ao horário e com (EAD) no nome; o resultado
# mostra apenas as encontradas nas fontes que não estão marcadas na coluna EAD.
validacao_ead = ead_concentrated_rows(planilha_df, fontes_completas_df)
display_result("B3) EAD não marcado na planilha", validacao_ead)


### B3) EAD não marcado na planilha

Nenhuma inconsistência encontrada.

In [14]:
# Regra B4: encontra turmas que não aparecem nos PDFs de curso e não estão marcadas como !DSC.
# A comparação usa Código + Nome contra os PDFs dos cursos; quando a turma não aparece lá,
# a planilha deve indicar que ela não pertence aos cursos acompanhados.
conflitos_flag_dsc = dsc_flag_conflicts(planilha_df, course_pdf_df).drop(columns=["Tipo de inconsistência"], errors="ignore")
display_result("B4) Conflito de !DSC: não marcado", conflitos_flag_dsc)


### B4) Conflito de !DSC: não marcado

,Fonte,Código,Nome,Curso,Professor,Horário
88,TurmasDSC_2026_2.numbers,SIS.0124.00.001-3,Gestão da Informação,ADM-N,Marcos Rodrigo Momo,Seg 14/15
1,TurmasDSC_2026_2.numbers,CMP.0175.01.002-7,Trabalho de Conclusão de Curso I,BCC-M,_OfertarSoNoite_,
83,TurmasDSC_2026_2.numbers,CMP.0200.00.001-8,Inteligência Artificial Aplicada à Engenharia,EIE-N,Marcelo Grafulha Vanti,Ter 12/13; Qua 12/13
43,TurmasDSC_2026_2.numbers,CMP.0193.00.001-1,Algoritmos e Programação,"EIE-N, EPR-N, EMC-N, ECV-N",Fábio Luis Perez,Ter 12/15
0,TurmasDSC_2026_2.numbers,SIS.0119.00.001-9,Optativa,SIS-N,,


In [15]:
# Regra B5: encontra turmas em laboratório sem a marcação correspondente na planilha.
# A regra confere se o espaço físico pertence à lista de laboratórios válidos e, nesse caso,
# exige que a coluna Laboratório esteja marcada.
conflitos_laboratorio = lab_conflicts(planilha_df)
display_result("B5) Conflito de Laboratório", conflitos_laboratorio)


### B5) Conflito de Laboratório

Nenhuma inconsistência encontrada.

In [16]:
# Regra C1: encontra turmas com total de créditos ímpar em todas as fontes.
# O total é calculado a partir dos créditos teóricos e práticos extraídos da planilha
# e dos PDFs, mantendo apenas as colunas relevantes para essa conferência.
validacao_creditos_impares = inconsistent_credit_rows(fontes_completas_df)
display_result("C1) Total de créditos ímpar", validacao_creditos_impares)


### C1) Total de créditos ímpar

,Fonte,Código,Nome,Curso,Teóricos,Práticos,Total
149,BCC_mat.pdf,CMP.0168.00.002-0,Programação Orientada a Objetos,BCC-M,7,0,7
153,BCC_mat.pdf,CMP.0170.02.002-4,Programação Web II,BCC-M,5,0,5
169,BCC_not.pdf,CMP.0166.00.001-4,Introdução à Programação,BCC-N,7,0,7
176,BCC_not.pdf,CMP.0168.00.001-1,Programação Orientada a Objetos,BCC-N,7,0,7
181,BCC_not.pdf,CMP.0168.00.004-6,Programação Orientada a Objetos,BCC-N,7,0,7
186,BCC_not.pdf,CMP.0170.01.001-0,Programação Web I,BCC-N,5,0,5
192,BCC_not.pdf,CMP.0170.01.002-9,Programação Web I,BCC-N,5,0,5
195,BCC_not.pdf,CMP.0170.02.001-6,Programação Web II,BCC-N,5,0,5
201,BCC_not.pdf,CMP.0170.02.003-2,Programação Web II,BCC-N,5,0,5
207,BCC_not.pdf,CMP.0170.02.004-0,Programação Web II,BCC-N,5,0,5


In [17]:
# Regra C2: lista turmas cuja carga horária informada no horário calculado ficou zero.
# A base de incompatibilidades é calculada uma vez e esta seção mantém somente os casos
# em que não há períodos de horário contabilizados.
validacao_horario_incompativel = schedule_credit_mismatch_rows(fontes_completas_df)
validacao_horario_zero = zero_schedule_credit_rows(validacao_horario_incompativel)
display_result("C2) Carga horária com zero", validacao_horario_zero)


### C2) Carga horária com zero

,Código,Nome,Fonte
0,ADM.0557.00.001-0,Administração Geral,SIS_not.pdf
1,CMP.0172.02.001-3,Eletiva II,"DSC.pdf, TurmasDSC_2026_2.numbers"
2,CMP.0172.04.001-4,Eletiva IV,"BCC_not.pdf, DSC.pdf, TurmasDSC_2026_2.numbers"
3,CMP.0172.05.001-0,Eletiva V,"BCC_not.pdf, DSC.pdf, TurmasDSC_2026_2.numbers"
4,CMP.0175.01.002-7,Trabalho de Conclusão de Curso I,TurmasDSC_2026_2.numbers
5,CMP.0180.00.003-4,Processamento de Linguagem Natural,"BCC_not.pdf, DSC.pdf"
6,CON.0157.00.001-3,Contabilidade Geral,SIS_not.pdf
7,DIR.0510.00.002-6,Legislação em Informática,TurmasDSC_2026_2.numbers
8,SIS.0105.00.001-1,Inovação Tecnológica,"DSC.pdf, TurmasDSC_2026_2.numbers"
9,SIS.0105.00.003-8,Inovação Tecnológica,"DSC.pdf, TurmasDSC_2026_2.numbers"


In [18]:
# Regra C3: lista turmas com carga horária incompatível com os créditos, exceto as de horário zero.
# A regra compara o total de créditos com os períodos extraídos do horário, agrupa duplicidades
# e mantém a coluna Fonte com todos os locais em que a inconsistência ocorreu.
validacao_horario_incompativel_sem_zero = nonzero_schedule_credit_mismatch_rows(validacao_horario_incompativel)
display_result("C3) Carga horária incompatível com créditos", validacao_horario_incompativel_sem_zero)


### C3) Carga horária incompatível com créditos

,Código,Nome,Horário informado,Créditos,Horário,Fonte
0,CDD.0002.00.001-0,Análise Exploratória de Dados,Ter 12/13; Ter 14/15,5,4,"BCD_not.pdf, DSC.pdf"
1,CDD.0002.00.002-8,Análise Exploratória de Dados,Qua 12/13; Qua 14/15,5,4,"BCD_not.pdf, DSC.pdf"
2,CMP.0166.00.001-4,Introdução à Programação,Seg 12/13; Qua 12/15,7,6,"BCC_not.pdf, DSC.pdf"
3,CMP.0168.00.001-1,Programação Orientada a Objetos,Seg 12/15; Qui 12/13,7,6,"BCC_not.pdf, DSC.pdf"
4,CMP.0168.00.002-0,Programação Orientada a Objetos,Qua 1/4; Qui 1/2,7,6,"BCC_mat.pdf, DSC.pdf"
5,CMP.0168.00.003-8,Programação Orientada a Objetos,Qua 12/15; Qui 12/13,7,6,"DSC.pdf, SIS_not.pdf"
6,CMP.0168.00.004-6,Programação Orientada a Objetos,Ter 12/15; Qui 14/15,7,6,"BCC_not.pdf, DSC.pdf"
7,CMP.0170.01.001-0,Programação Web I,Qua 12/15,5,4,"BCC_not.pdf, DSC.pdf"
8,CMP.0170.01.002-9,Programação Web I,Ter 12/15,5,4,"BCC_not.pdf, DSC.pdf"
9,CMP.0170.02.001-6,Programação Web II,Ter 12/15,5,4,"BCC_not.pdf, DSC.pdf"


In [19]:
# Regra D1: lista turmas sem professor em todas as fontes.
# Turmas repetidas são agrupadas por Código + Nome, a coluna Fonte reúne todos os locais
# em que ocorreram, e códigos iniciados por PDE. são ignorados.
validacao_sem_professor = rows_without_professor(fontes_completas_df)
display_result("D1) Turmas sem professor", validacao_sem_professor)


### D1) Turmas sem professor

,Código,Nome,Fonte
0,CMP.0084.00.001-0,Introdução à Computação,"BCC_not.pdf, DSC.pdf"
1,CMP.0169.00.001-5,Lógica para Computação,"BCC_not.pdf, DSC.pdf"
2,CMP.0169.00.002-3,Lógica para Computação,"BCC_mat.pdf, DSC.pdf"
3,CMP.0169.00.003-1,Lógica para Computação,"DSC.pdf, SIS_not.pdf"
4,CMP.0169.00.004-0,Lógica para Computação,"BCC_not.pdf, DSC.pdf"
5,CMP.0169.00.005-8,Lógica para Computação,"BCD_not.pdf, DSC.pdf"
6,CMP.0169.00.006-6,Lógica para Computação,"BCD_not.pdf, DSC.pdf"
7,CMP.0170.01.002-9,Programação Web I,"BCC_not.pdf, DSC.pdf"
8,CMP.0170.02.001-6,Programação Web II,"BCC_not.pdf, DSC.pdf"
9,CMP.0170.02.003-2,Programação Web II,"BCC_not.pdf, DSC.pdf"


In [20]:
# Regra D2: encontra diferenças de professor entre a planilha e os PDFs para o mesmo código.
# A comparação usa igualdade direta, preservando diferenças de acento, caixa e espaçamento.
# Linhas com professor em branco em qualquer fonte ficam fora desta regra, pois entram na D1.
display_result("D2) Professor diferente", professor_diferente)


### D2) Professor diferente

,Código,Nome Planilha,Nome PDF,Professor Planilha,Professor PDF,Horário Planilha,Horário PDF,Curso Planilha,Curso PDF,Fonte PDF
46,MAT.0193.00.002-0,Geometria Analítica,Geometria Analítica,Christiano Garcia,Simone Leal Schwertl,Qua 1/4,Qua 1/4,BCC-M,BCC-M,BCC_mat.pdf
37,CMP.0170.02.004-0,Programação Web II,Programação Web II,Bruno Fischer Ferreira Santos,Luciana Pereira de Araújo Kohler,Seg 12/15,Seg 12/15,BCC-N,BCC-N,BCC_not.pdf
92,CMP.0190.00.003-7,Redes de Computadores,Redes de Computadores,Guilherme Legal de Oliveira,Hélio Ricardo Naumann,Seg 12/15,Seg 12/15,BCC-N,BCC-N,BCC_not.pdf
111,MAT.0193.00.003-8,Geometria Analítica,Geometria Analítica,José Carlos Althoff,Simone Leal Schwertl,Ter 12/15,Ter 12/15,BCC-N,BCC-N,BCC_not.pdf
43,MAT.0219.00.001-3,Álgebra Linear,Álgebra Linear,Christiano Garcia,Simone Leal Schwertl,Qua 12/15,Qua 12/15,BCC-N,BCC-N,BCC_not.pdf
48,SIS.0110.00.001-6,Engenharia de Software,Engenharia de Software,Cláudia Neli de Souza Zambon,Ricardo Voigt,Ter 12/15,Ter 12/15,BCC-N,BCC-N,BCC_not.pdf
38,CMP.0170.02.004-0,Programação Web II,Programação Web II,Bruno Fischer Ferreira Santos,Luciana Pereira de Araújo Kohler,Seg 12/15,Seg 12/15,BCC-N,BCC-N,DSC.pdf
93,CMP.0190.00.003-7,Redes de Computadores,Redes de Computadores,Guilherme Legal de Oliveira,Hélio Ricardo Naumann,Seg 12/15,Seg 12/15,BCC-N,BCC-N,DSC.pdf
49,SIS.0110.00.001-6,Engenharia de Software,Engenharia de Software,Cláudia Neli de Souza Zambon,Ricardo Voigt,Ter 12/15,Ter 12/15,BCC-N,BCC-N,DSC.pdf
148,SIS.0124.00.001-3,Gestão da Informação,Gestão da Informação,Marcos Rodrigo Momo,Angelica Karize Viecelli,Seg 14/15,Seg 14/15,ADM-N,ADM-N,DSC.pdf


In [21]:
# Regra D3: encontra conflitos de professor na planilha.
# Uma inconsistência ocorre quando o mesmo professor aparece em duas ou mais turmas
# no mesmo dia e período de horário.
conflitos_professor = find_conflicts(
    planilha_df,
    ["Professor"],
    {"Professor": "Professor"},
    "Professor em duas ou mais turmas",
).drop(columns=["Tipo de conflito"], errors="ignore")
display_result("D3) Conflitos de professor: duas ou mais turmas", conflitos_professor)


### D3) Conflitos de professor: duas ou mais turmas

,Professor,Horário,Código,Nome,Curso
0,Alexander Roberto Valdameri,Seg 12,SIS.0125.00.001-7,Banco de Dados I,BCD-N
1,Alexander Roberto Valdameri,Seg 12,SIS.0125.00.002-5,Banco de Dados I,BCD-N
2,Alexander Roberto Valdameri,Seg 13,SIS.0125.00.001-7,Banco de Dados I,BCD-N
3,Alexander Roberto Valdameri,Seg 13,SIS.0125.00.002-5,Banco de Dados I,BCD-N
4,Alexander Roberto Valdameri,Seg 14,SIS.0125.00.001-7,Banco de Dados I,BCD-N
5,Alexander Roberto Valdameri,Seg 14,SIS.0125.00.002-5,Banco de Dados I,BCD-N
6,Alexander Roberto Valdameri,Seg 15,SIS.0125.00.001-7,Banco de Dados I,BCD-N
7,Alexander Roberto Valdameri,Seg 15,SIS.0125.00.002-5,Banco de Dados I,BCD-N
8,Anne Caroline Peixer Abreu Neves,Sex 12,HIS.0116.00.002-4,História da Cultura Afro-brasileira e Indígena (EAD),BCC-N
9,Anne Caroline Peixer Abreu Neves,Sex 13,HIS.0116.00.002-4,História da Cultura Afro-brasileira e Indígena (EAD),BCC-N


In [22]:
# Regra E1: encontra conflitos de espaço físico na planilha.
# Uma inconsistência ocorre quando o mesmo espaço aparece em duas ou mais turmas
# no mesmo dia e período de horário.
conflitos_espaco = find_conflicts(
    planilha_df,
    ["Espaço"],
    {"Espaço": "Espaço"},
    "Espaço em duas ou mais turmas",
).drop(columns=["Tipo de conflito"], errors="ignore")
display_result("E1) Conflitos de espaço físico: duas ou mais turmas", conflitos_espaco)


### E1) Conflitos de espaço físico: duas ou mais turmas

,Espaço,Horário,Código,Nome,Curso
0,S-401,Qua 12,CDD.0002.00.002-8,Análise Exploratória de Dados,BCD-N
1,S-401,Qua 12,MAT.0253.00.001-2,Fundamentos de Matemática,BCD-N
2,S-401,Qua 13,CDD.0002.00.002-8,Análise Exploratória de Dados,BCD-N
3,S-401,Qua 13,MAT.0253.00.001-2,Fundamentos de Matemática,BCD-N
4,S-401,Qua 14,CDD.0002.00.002-8,Análise Exploratória de Dados,BCD-N
5,S-401,Qua 14,MAT.0253.00.001-2,Fundamentos de Matemática,BCD-N
6,S-401,Qua 15,CDD.0002.00.002-8,Análise Exploratória de Dados,BCD-N
7,S-401,Qua 15,MAT.0253.00.001-2,Fundamentos de Matemática,BCD-N
8,S-401,Seg 12,SIS.0125.00.001-7,Banco de Dados I,BCD-N
9,S-401,Seg 12,SIS.0125.00.002-5,Banco de Dados I,BCD-N


In [23]:
# Regra F1: encontra conflitos de semestre/fase/grupo na planilha.
# Uma inconsistência ocorre quando o mesmo Curso + Fase + Grupo tem duas ou mais turmas
# no mesmo dia e período de horário.
conflitos_fase = find_conflicts(
    planilha_df,
    ["Curso", "Fase", "Grupo"],
    {"Curso": "Curso", "Fase": "Fase", "Grupo": "Grupo"},
    "Mesmo curso/fase/grupo no mesmo horário",
).drop(columns=["Tipo de conflito"], errors="ignore")
display_result("F1) Conflitos de semestre/fase: mesmo curso/fase/grupo no mesmo horário", conflitos_fase)


### F1) Conflitos de semestre/fase: mesmo curso/fase/grupo no mesmo horário

,Curso,Fase,Grupo,Horário,Código,Nome
0,BCC-N,4.0,A,Sex 12,HIS.0116.00.002-4,História da Cultura Afro-brasileira e Indígena (EAD)
1,BCC-N,4.0,A,Sex 12,SOC.0200.00.009-6,Alteridade e Direitos Humanos (EAD)
2,BCC-N,4.0,A,Sex 13,HIS.0116.00.002-4,História da Cultura Afro-brasileira e Indígena (EAD)
3,BCC-N,4.0,A,Sex 13,SOC.0200.00.009-6,Alteridade e Direitos Humanos (EAD)
4,BCC-N,4.0,B,Sex 14,HIS.0116.00.010-5,História da Cultura Afro-brasileira e Indígena (EAD)
5,BCC-N,4.0,B,Sex 14,SOC.0200.00.002-9,Alteridade e Direitos Humanos (EAD)
6,BCC-N,4.0,B,Sex 15,HIS.0116.00.010-5,História da Cultura Afro-brasileira e Indígena (EAD)
7,BCC-N,4.0,B,Sex 15,SOC.0200.00.002-9,Alteridade e Direitos Humanos (EAD)


In [24]:
# Geração do relatório final.
# Esta célula consolida as quantidades de todas as regras, monta o resumo agrupado,
# monta as seções Markdown na mesma ordem das células e grava comparacao.md em Dados.
summary_groups = [
    ("A) __ Conflitos entre PDFs e Planilha (qtd.)", [
        {"Tipo": "A1) Turmas dos cursos PDFs ausentes na planilha", "Quantidade": len(ausentes_cursos)},
        {"Tipo": "A2) Turmas do DSC ausentes na planilha", "Quantidade": len(ausentes_dsc)},
        {"Tipo": "A3) Horário diferente", "Quantidade": len(horario_diferente)},
        {"Tipo": "A4) Nome diferente", "Quantidade": len(nome_diferente)},
        {"Tipo": "A5) Curso diferente", "Quantidade": len(curso_diferente)},
    ]),
    ("B) __ Coluna Caixa de Seleção (qtd.)", [
        {"Tipo": "B1) Concentrado não marcado na planilha", "Quantidade": len(validacao_concentrado)},
        {"Tipo": "B2) Conflito de Concentrado", "Quantidade": len(conflitos_flag_concentrado)},
        {"Tipo": "B3) EAD não marcado na planilha", "Quantidade": len(validacao_ead)},
        {"Tipo": "B4) Conflito de !DSC: não marcado", "Quantidade": len(conflitos_flag_dsc)},
        {"Tipo": "B5) Conflito de Laboratório", "Quantidade": len(conflitos_laboratorio)},
    ]),
    ("C) __ Conflitos créditos x horas aula (qtd.)", [
        {"Tipo": "C1) Total de créditos ímpar", "Quantidade": len(validacao_creditos_impares)},
        {"Tipo": "C2) Carga horária com zero", "Quantidade": len(validacao_horario_zero)},
        {"Tipo": "C3) Carga horária incompatível com créditos", "Quantidade": len(validacao_horario_incompativel_sem_zero)},
    ]),
    ("D) __ Conflitos Professor (qtd.)", [
        {"Tipo": "D1) Turmas sem professor", "Quantidade": len(validacao_sem_professor)},
        {"Tipo": "D2) Professor diferente", "Quantidade": len(professor_diferente)},
        {"Tipo": "D3) Conflitos de professor: duas ou mais turmas", "Quantidade": len(conflitos_professor)},
    ]),
    ("E) __ Conflitos Espaço físico (qtd.)", [
        {"Tipo": "E1) Conflitos de espaço físico: duas ou mais turmas", "Quantidade": len(conflitos_espaco)},
    ]),
    ("F) __ Conflitos Horário no Semestre (qtd.)", [
        {"Tipo": "F1) Conflitos de semestre/fase: mesmo curso/fase/grupo no mesmo horário", "Quantidade": len(conflitos_fase)},
    ]),
]

summary_df = pd.concat(
    [pd.DataFrame(rows).assign(Grupo=title) for title, rows in summary_groups],
    ignore_index=True,
)
summary_df = summary_df[["Grupo", "Tipo", "Quantidade"]]

summary_parts = ["## Resumo", ""]
for group_title, rows in summary_groups:
    summary_parts.append(markdown_section(f"### {group_title}", pd.DataFrame(rows)))
summary_markdown = "\n".join(summary_parts).strip() + "\n"

report_parts = [
    "# Relatório de Comparação de Dados",
    "",
    summary_markdown,
    "## A) __ Conflitos entre PDFs e Planilha\n",
    markdown_section("### A1) Turmas dos cursos PDFs ausentes na planilha", ausentes_cursos),
    markdown_section("### A2) Turmas do DSC ausentes na planilha", ausentes_dsc),
    markdown_section("### A3) Horário diferente", horario_diferente),
    markdown_section("### A4) Nome diferente", nome_diferente),
    markdown_section("### A5) Curso diferente", curso_diferente),
    "## B) __ Coluna Caixa de Seleção\n",
    markdown_section("### B1) Concentrado não marcado na planilha", validacao_concentrado),
    markdown_section("### B2) Conflito de Concentrado", conflitos_flag_concentrado),
    markdown_section("### B3) EAD não marcado na planilha", validacao_ead),
    markdown_section("### B4) Conflito de !DSC: não marcado", conflitos_flag_dsc),
    markdown_section("### B5) Conflito de Laboratório", conflitos_laboratorio),
    "## C) __ Conflitos créditos x horas aula\n",
    markdown_section("### C1) Total de créditos ímpar", validacao_creditos_impares),
    markdown_section("### C2) Carga horária com zero", validacao_horario_zero),
    markdown_section("### C3) Carga horária incompatível com créditos", validacao_horario_incompativel_sem_zero),
    "## D) __ Conflitos Professor\n",
    markdown_section("### D1) Turmas sem professor", validacao_sem_professor),
    markdown_section("### D2) Professor diferente", professor_diferente),
    markdown_section("### D3) Conflitos de professor: duas ou mais turmas", conflitos_professor),
    "## E) __ Conflitos Espaço físico\n",
    markdown_section("### E1) Conflitos de espaço físico: duas ou mais turmas", conflitos_espaco),
    "## F) __ Conflitos Horário no Semestre\n",
    markdown_section("### F1) Conflitos de semestre/fase: mesmo curso/fase/grupo no mesmo horário", conflitos_fase),
]

report_text = "\n".join(report_parts).strip() + "\n"
report_path = dados_dir / "comparacao.md"
report_path.write_text(report_text, encoding="utf-8")

print(f"Relatório salvo em: {report_path}")
display(Markdown(summary_markdown))
display(Markdown(report_text[:4000] + ("\n\n..." if len(report_text) > 4000 else "")))


Relatório salvo em: /Users/daltonreis/Library/Mobile Documents/com~apple~Numbers/Documents/DSC/2026_2/2026-06-04/comparacao.md


## Resumo

### A) __ Conflitos entre PDFs e Planilha (qtd.)

| Tipo                                            |   Quantidade |
|:------------------------------------------------|-------------:|
| A1) Turmas dos cursos PDFs ausentes na planilha |            0 |
| A2) Turmas do DSC ausentes na planilha          |            0 |
| A3) Horário diferente                           |           18 |
| A4) Nome diferente                              |            0 |
| A5) Curso diferente                             |           15 |

### B) __ Coluna Caixa de Seleção (qtd.)

| Tipo                                    |   Quantidade |
|:----------------------------------------|-------------:|
| B1) Concentrado não marcado na planilha |            0 |
| B2) Conflito de Concentrado             |            0 |
| B3) EAD não marcado na planilha         |            0 |
| B4) Conflito de !DSC: não marcado       |            5 |
| B5) Conflito de Laboratório             |            0 |

### C) __ Conflitos créditos x horas aula (qtd.)

| Tipo                                        |   Quantidade |
|:--------------------------------------------|-------------:|
| C1) Total de créditos ímpar                 |           30 |
| C2) Carga horária com zero                  |           12 |
| C3) Carga horária incompatível com créditos |           39 |

### D) __ Conflitos Professor (qtd.)

| Tipo                                            |   Quantidade |
|:------------------------------------------------|-------------:|
| D1) Turmas sem professor                        |           20 |
| D2) Professor diferente                         |           10 |
| D3) Conflitos de professor: duas ou mais turmas |           20 |

### E) __ Conflitos Espaço físico (qtd.)

| Tipo                                                |   Quantidade |
|:----------------------------------------------------|-------------:|
| E1) Conflitos de espaço físico: duas ou mais turmas |           24 |

### F) __ Conflitos Horário no Semestre (qtd.)

| Tipo                                                                    |   Quantidade |
|:------------------------------------------------------------------------|-------------:|
| F1) Conflitos de semestre/fase: mesmo curso/fase/grupo no mesmo horário |            8 |


# Relatório de Comparação de Dados

## Resumo

### A) __ Conflitos entre PDFs e Planilha (qtd.)

| Tipo                                            |   Quantidade |
|:------------------------------------------------|-------------:|
| A1) Turmas dos cursos PDFs ausentes na planilha |            0 |
| A2) Turmas do DSC ausentes na planilha          |            0 |
| A3) Horário diferente                           |           18 |
| A4) Nome diferente                              |            0 |
| A5) Curso diferente                             |           15 |

### B) __ Coluna Caixa de Seleção (qtd.)

| Tipo                                    |   Quantidade |
|:----------------------------------------|-------------:|
| B1) Concentrado não marcado na planilha |            0 |
| B2) Conflito de Concentrado             |            0 |
| B3) EAD não marcado na planilha         |            0 |
| B4) Conflito de !DSC: não marcado       |            5 |
| B5) Conflito de Laboratório             |            0 |

### C) __ Conflitos créditos x horas aula (qtd.)

| Tipo                                        |   Quantidade |
|:--------------------------------------------|-------------:|
| C1) Total de créditos ímpar                 |           30 |
| C2) Carga horária com zero                  |           12 |
| C3) Carga horária incompatível com créditos |           39 |

### D) __ Conflitos Professor (qtd.)

| Tipo                                            |   Quantidade |
|:------------------------------------------------|-------------:|
| D1) Turmas sem professor                        |           20 |
| D2) Professor diferente                         |           10 |
| D3) Conflitos de professor: duas ou mais turmas |           20 |

### E) __ Conflitos Espaço físico (qtd.)

| Tipo                                                |   Quantidade |
|:----------------------------------------------------|-------------:|
| E1) Conflitos de espaço físico: duas ou mais turmas |           24 |

### F) __ Conflitos Horário no Semestre (qtd.)

| Tipo                                                                    |   Quantidade |
|:------------------------------------------------------------------------|-------------:|
| F1) Conflitos de semestre/fase: mesmo curso/fase/grupo no mesmo horário |            8 |

## A) __ Conflitos entre PDFs e Planilha

### A1) Turmas dos cursos PDFs ausentes na planilha

Nenhuma inconsistência encontrada.

### A2) Turmas do DSC ausentes na planilha

Nenhuma inconsistência encontrada.

### A3) Horário diferente

| Código            | Nome Planilha                      | Nome PDF                           | Professor Planilha                 | Professor PDF                      | Horário Planilha   | Horário PDF                                                     | Curso Planilha   | Curso PDF   | Fonte PDF   |
|:------------------|:-----------------------------------|:-----------------------------------|:-----------------------------------|:-----------------------------------|:-------------------|:----------------------------------------------------------------|:-----------------|:------------|:------------|
| CMP.0172.02.001-3 | Eletiva II                         | Eletiva II                         | Leandro Werner Ribeiro             | Leandro Werner Ribeiro             |                    | Seg 12/15; Ter 12/15; Qua 12/15; Qui 12/15; Sex 12/15C          | BCC-N            | BCC-N       | BCC_not.pdf |
| CMP.0180.00.003-4 | Processamento de Linguagem Natural | Processamento de Linguagem Natural | Maiko Rafael Spiess                | Maiko Rafael Spiess                | Qui 12/15          |                                                                 | BCC-N            | BCC-N       | BCC_not.pdf |
| SIS.0105.00.001-1 | Inovação Tecnológica               | Inovação Tecnológica               | Simone Erbs da Costa               | Simone Erbs da Costa               |                    | Seg 12/15; Ter

...